# Advanced EDA: Pakistan Weather, 2020-2025
Source: `daily_weather` table in `weather.db` (6 cities, daily values).

In [ ]:
import os
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

DB = os.environ.get("WEATHER_DB", "../data/weather.db")
with sqlite3.connect(DB) as con:
    df = pd.read_sql("SELECT * FROM daily_weather", con, parse_dates=["date"])

df["temp_avg"] = (df.temp_max + df.temp_min) / 2
df["year"], df["month"] = df.date.dt.year, df.date.dt.month
df["season"] = df.month.map({12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                             6: "Summer", 7: "Summer", 8: "Summer"}).fillna("Autumn")

print(df.shape, "| missing values:", df.isna().sum().sum())
df.describe().round(1)

## Q1. Which city is hottest and which is coldest on average?

In [ ]:
by_city = df.groupby("city").temp_avg.mean().sort_values()
by_city.plot.barh(color="tomato", title="Average temperature by city (°C)")
print(f"Hottest: {by_city.idxmax()} ({by_city.max():.1f} °C) | Coldest: {by_city.idxmin()} ({by_city.min():.1f} °C)")

**Insight:** _write 1-2 lines after running on the real data._

## Q2. How does temperature change over time? (monthly trend and yearly average)

In [ ]:
monthly = df.set_index("date").groupby("city").temp_avg.resample("MS").mean().unstack(0)
monthly.plot(figsize=(11, 4), title="Monthly average temperature (°C)")
plt.show()
df.pivot_table(index="year", columns="city", values="temp_avg").round(1)

**Insight:** _write 1-2 lines after running on the real data._

## Q3. Which month receives the most rain in each city?

In [ ]:
rain = df.groupby(["city", "year", "month"]).precipitation.sum().groupby(["city", "month"]).mean().unstack(0)
sns.heatmap(rain.T, annot=True, fmt=".0f", cmap="Blues", cbar_kws={"label": "mm per month (yearly average)"})
plt.title("Average monthly rainfall")
print(rain.idxmax().rename("rainiest month"))

**Insight:** _write 1-2 lines after running on the real data._

## Q4. Is temperature correlated with rainfall?
Spearman is used because daily rainfall is heavily skewed (many zero days).

In [ ]:
sns.heatmap(df[["temp_max", "temp_min", "precipitation", "wind"]].rank().corr(),
            annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Spearman correlation (all cities, daily)")

**Insight:** _write 1-2 lines after running on the real data._

## Q5. How do summer and winter compare across cities?

In [ ]:
sns.barplot(data=df[df.season.isin(["Summer", "Winter"])], x="city", y="temp_avg", hue="season", errorbar=None)
plt.title("Average temperature: summer vs winter (°C)")

**Insight:** _write 1-2 lines after running on the real data._